# Mission_Analysis_Test_File
# Generalized pipeline for cubesat post-deployment identification analysis

## Checking the discriminatory strength of BSTAR as a discriminator for the identification and elimination of candidate satellites. 

# Imports 

In [15]:
import numpy as np
import pandas as pd
import json
import os
from datetime import datetime, timezone
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, CartesianRepresentation
import astropy.units as u
from sgp4.api import Satrec, SatrecArray

# Constants

In [16]:
ST_BASE  = "https://www.space-track.org"
ST_LOGIN = f"{ST_BASE}/ajaxauth/login"
ST_QUERY = f"{ST_BASE}/basicspacedata/query"

CACHE_DIR = os.path.expanduser("~/mission_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

# Authentication

In [17]:
def login(username, password, session):
    """Login to Space-Track and return session."""
    resp = session.post(ST_LOGIN, data={"identity": "nathandarby2022@gmail.com", "password": "Boobear32*Boobear32"})
    if resp.status_code == 200:
        print(f"Logged in to Space-Track successfully!")
    else:
        print(f"Login failed: {resp.status_code}")
    return session

# TLE Fetching

In [18]:
def fetch_norad_ids(intldes, session, force=False):
    """Fetch NORAD IDs for a launch group from satcat. Cached permanently."""
    cache_file = os.path.join(CACHE_DIR, f"{intldes}_satcat.json")
    
    if os.path.exists(cache_file) and not force:
        with open(cache_file) as f:
            cache = json.load(f)
        print(f"SATCAT loaded from cache — {len(cache['norad_ids'])} objects")
        return cache['norad_ids']
    
    print(f"Querying satcat for {intldes}...")
    url = (f"{ST_QUERY}/class/satcat"
           f"/INTLDES/{intldes}~~"
           f"/OBJECT_TYPE/Payload"
           f"/orderby/NORAD_CAT_ID"
           f"/format/json")
    
    resp = session.get(url)
    if resp.status_code != 200:
        print(f"satcat query failed: {resp.status_code}")
        return []
    
    objects   = resp.json()
    norad_ids = [obj['NORAD_CAT_ID'] for obj in objects]
    
    cache = {
        'intldes':    intldes,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'norad_ids':  norad_ids,
        'objects':    [{'NORAD_CAT_ID': obj['NORAD_CAT_ID'],
                        'SATNAME':      obj['SATNAME']} for obj in objects]
    }
    with open(cache_file, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Fetched {len(norad_ids)} payloads for {intldes}")
    return norad_ids


def fetch_current_tles(intldes, norad_ids, session, chunk_size=50, force=False):
    """Fetch current TLEs from gp class. Cached for 1 hour."""
    cache_file = os.path.join(CACHE_DIR, f"{intldes}_tles.json")
    
    if os.path.exists(cache_file) and not force:
        with open(cache_file) as f:
            cache = json.load(f)
        fetched_at = datetime.fromisoformat(cache['fetched_at'])
        age = datetime.now(timezone.utc) - fetched_at
        if age.total_seconds() < 3600:
            print(f"TLEs loaded from cache — {len(cache['tles'])} objects")
            return cache['tles']
        print("TLE cache expired — fetching fresh")
    
    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk   = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)
        url = (f"{ST_QUERY}/class/gp"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/decay_date/null-val"
               f"/orderby/NORAD_CAT_ID"
               f"/format/json")
        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code}")
        if resp.status_code != 200 or not resp.text.strip():
            continue
        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })
    
    cache = {'fetched_at': datetime.now(timezone.utc).isoformat(), 'tles': all_tles}
    with open(cache_file, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Fetched {len(all_tles)} TLEs for {intldes}")
    return all_tles


def fetch_historical_tles(intldes, norad_ids, launch_date, session,
                           days=60, chunk_size=50, force=False):
    """Fetch historical TLEs from gp_history. Cached permanently."""
    cache_file = os.path.join(CACHE_DIR, f"{intldes}_history.json")
    
    if os.path.exists(cache_file) and not force:
        with open(cache_file) as f:
            cache = json.load(f)
        print(f"Historical TLEs loaded from cache — {len(cache['tles'])} records")
        return cache['tles']
    
    # Parse launch date
    launch_dt  = pd.Timestamp(launch_date)
    end_date   = (launch_dt + pd.Timedelta(days=days)).strftime('%Y-%m-%d')
    start_date = launch_dt.strftime('%Y-%m-%d')
    
    print(f"Querying gp_history: {start_date} → {end_date}")
    print(f"WARNING: One-time query — caching permanently")
    
    all_tles = []
    for i in range(0, len(norad_ids), chunk_size):
        chunk   = norad_ids[i:i+chunk_size]
        ids_str = ','.join(str(n) for n in chunk)
        url = (f"{ST_QUERY}/class/gp_history"
               f"/NORAD_CAT_ID/{ids_str}"
               f"/epoch/{start_date}--{end_date}"
               f"/orderby/NORAD_CAT_ID,EPOCH"
               f"/format/json")
        resp = session.get(url)
        print(f"   Chunk {i//chunk_size + 1}: status {resp.status_code} "
              f"— {len(resp.json()) if resp.status_code == 200 else 0} records")
        if resp.status_code != 200 or not resp.text.strip():
            continue
        for obj in resp.json():
            if obj.get('TLE_LINE1') and obj.get('TLE_LINE2'):
                all_tles.append({
                    'name':  obj['OBJECT_NAME'],
                    'norad': obj['NORAD_CAT_ID'],
                    'epoch': obj['EPOCH'],
                    'line1': obj['TLE_LINE1'],
                    'line2': obj['TLE_LINE2']
                })
    
    cache = {
        'intldes':    intldes,
        'start_date': start_date,
        'end_date':   end_date,
        'fetched_at': datetime.now(timezone.utc).isoformat(),
        'tles':       all_tles
    }
    with open(cache_file, 'w') as f:
        json.dump(cache, f, indent=2)
    
    print(f"Cached {len(all_tles)} historical records")
    return all_tles


# Propagation

In [19]:
def propagate_tles(tles, duration_hrs=24, num_steps=2000):
    """Vectorized SGP4 propagation for all TLEs."""
    now    = Time(datetime.now(timezone.utc))
    times  = now + np.linspace(0, duration_hrs * 3600, num_steps) * u.s
    
    sats   = SatrecArray([Satrec.twoline2rv(t['line1'], t['line2']) for t in tles])
    jd1    = np.array([t.jd1 for t in times])
    jd2    = np.array([t.jd2 for t in times])
    
    e, r_teme, v_teme = sats.sgp4(jd1, jd2)
    
    # Convert to lat/lon/alt
    lats = np.zeros((len(tles), num_steps))
    lons = np.zeros((len(tles), num_steps))
    alts = np.zeros((len(tles), num_steps))
    
    for i in range(len(tles)):
        for j in range(num_steps):
            if e[i, j] != 0:
                continue
            r     = CartesianRepresentation(r_teme[i, j] * u.km)
            teme  = TEME(r, obstime=times[j])
            itrs  = teme.transform_to(ITRS(obstime=times[j]))
            loc   = EarthLocation(*itrs.cartesian.xyz)
            lats[i, j] = loc.lat.deg
            lons[i, j] = loc.lon.deg
            alts[i, j] = loc.height.to(u.km).value
    
    print(f"Propagation complete — {len(tles)} satellites, {num_steps} steps")
    print(f"   Errors: {(e != 0).sum()} non-zero error codes")
    return e, r_teme, v_teme, lats, lons, alts, times

# TLE Dataframe

In [20]:
def parse_bstar(s):
    """Parse compressed scientific notation BSTAR from TLE."""
    s = s.strip()
    for i in range(1, len(s)):
        if s[i] in '+-':
            mantissa = float('0.' + s[:i])
            exp      = int(s[i:])
            return mantissa * (10 ** exp)
    return float(s)

def build_tle_dataframe(tles, alts, e):
    """Build analytical dataframe from TLEs and propagated data."""
    rows = []
    for i, t in enumerate(tles):
        line1 = t['line1']
        line2 = t['line2']
        
        intldes_raw = line1[9:17].strip()
        year        = intldes_raw[:2]
        num         = intldes_raw[2:5]
        piece       = intldes_raw[5:]
        full_year   = f"19{year}" if int(year) >= 57 else f"20{year}"
        intldes     = f"{full_year}-{num}{piece}"
        
        valid = alts[i][e[i] == 0]
        
        rows.append({
            'idx':          i,
            'name':         t['name'],
            'intldes':      intldes,
            'norad':        line1[2:7].strip(),
            'bstar':        parse_bstar(line1[53:61]),
            'mean_motion':  float(line2[52:63].strip()),
            'inclination':  float(line2[8:16].strip()),
            'eccentricity': float('0.' + line2[26:33].strip()),
            'raan':         float(line2[17:25].strip()),
            'arg_perigee':  float(line2[34:42].strip()),
            'mean_anomaly': float(line2[43:51].strip()),
            'alt_mean':     valid.mean() if len(valid) > 0 else np.nan,
            'alt_min':      valid.min()  if len(valid) > 0 else np.nan,
            'alt_max':      valid.max()  if len(valid) > 0 else np.nan,
            'period_min':   1440 / float(line2[52:63].strip())
        })
    
    df = pd.DataFrame(rows)
    df['alt_range']        = df['alt_max'] - df['alt_min']
    df['decay_rate_km_day'] = [
        np.polyfit(np.linspace(0, 24, alts.shape[1])[e[i] == 0],
                   alts[i][e[i] == 0], 1)[0] * 24
        if (e[i] == 0).sum() > 10 else np.nan
        for i in range(len(tles))
    ]
    return df

# Clustering 

In [21]:
def cluster_by_altitude(df):
    """Simple altitude-based clustering."""
    def assign(alt):
        if alt < 300:
            return 'A — Decaying (<300km)'
        elif alt < 400:
            return 'B — Lower Core (300-400km)'
        elif alt < 480:
            return 'C — Upper Core (400-480km)'
        else:
            return 'D — High Shell (>480km)'
    
    df['cluster'] = df['alt_mean'].apply(assign)
    
    for cluster, group in df.groupby('cluster'):
        print(f"{cluster} — {len(group)} satellites "
              f"({group['alt_mean'].min():.0f}-{group['alt_mean'].max():.0f} km)")
    return df

# BSTAR Correlation

In [22]:
def compute_bstar_correlation(df, exclude_maneuvering=True):
    """Compute BSTAR vs decay rate correlation."""
    df_clean = df.copy()
    
    if exclude_maneuvering:
        df_clean = df_clean[df_clean['alt_range'] < 50]
        print(f"Excluded {len(df) - len(df_clean)} maneuvering satellites")
    
    corr_raw = df_clean['bstar'].corr(df_clean['decay_rate_km_day'])
    r2       = corr_raw ** 2
    
    print(f"BSTAR vs decay rate correlation:")
    print(f"   Pearson r:  {corr_raw:.4f}")
    print(f"   R²:         {r2:.4f}")
    print(f"   Explained variance: {r2*100:.1f}%")
    
    return corr_raw, r2

# RTN Separation

In [23]:
def build_hist_dataframe(historical_tles, launch_date):
    """Build historical TLE dataframe."""
    hist_df = pd.DataFrame(historical_tles)
    hist_df['epoch'] = pd.to_datetime(hist_df['epoch'], utc=True)
    hist_df['date']  = hist_df['epoch'].dt.date
    hist_df['days_since_launch'] = (
        hist_df['epoch'] - pd.Timestamp(launch_date, tz='UTC')
    ).dt.total_seconds() / 86400
    return hist_df


def compute_rtn_separation(hist_df, launch_date, min_sats=5):
    """Compute RTN pairwise separation over time."""
    daily_tles  = (hist_df.sort_values('epoch')
                          .groupby(['norad', 'date'])
                          .first()
                          .reset_index())
    unique_days = sorted(daily_tles['date'].unique())
    
    rtn_results = []
    
    for day in unique_days:
        day_tles = daily_tles[daily_tles['date'] == day]
        day_tles = day_tles[day_tles['line1'].notna() & day_tles['line2'].notna()]
        
        if len(day_tles) < min_sats:
            continue
        
        eval_time = Time(f"{day}T12:00:00", scale='utc')
        jd1, jd2  = eval_time.jd1, eval_time.jd2
        
        positions  = []
        velocities = []
        
        for _, row in day_tles.iterrows():
            try:
                sat     = Satrec.twoline2rv(row['line1'], row['line2'])
                e, r, v = sat.sgp4(jd1, jd2)
                if e == 0:
                    positions.append(np.array(r))
                    velocities.append(np.array(v))
            except:
                continue
        
        if len(positions) < min_sats:
            continue
        
        positions  = np.array(positions)
        velocities = np.array(velocities)
        
        r_seps, t_seps, n_seps = [], [], []
        
        for a in range(len(positions)):
            R_hat = positions[a] / np.linalg.norm(positions[a])
            N_hat = np.cross(positions[a], velocities[a])
            N_hat = N_hat / np.linalg.norm(N_hat)
            T_hat = np.cross(N_hat, R_hat)
            T_hat = T_hat / np.linalg.norm(T_hat)
            
            for b in range(a+1, len(positions)):
                dr = positions[b] - positions[a]
                r_seps.append(np.abs(np.dot(dr, R_hat)))
                t_seps.append(np.abs(np.dot(dr, T_hat)))
                n_seps.append(np.abs(np.dot(dr, N_hat)))
        
        days_since = (pd.Timestamp(day) - pd.Timestamp(launch_date)).days
        
        rtn_results.append({
            'date':              day,
            'days_since_launch': days_since,
            'n_sats':            len(positions),
            'mean_R':            np.mean(r_seps),
            'mean_T':            np.mean(t_seps),
            'mean_N':            np.mean(n_seps),
            'median_R':          np.median(r_seps),
            'median_T':          np.median(t_seps),
            'median_N':          np.median(n_seps),
        })
    
    rtn_df = pd.DataFrame(rtn_results)
    rtn_df['rsi_R'] = rtn_df['mean_R'] / rtn_df['mean_R'].iloc[0]
    rtn_df['rsi_T'] = rtn_df['mean_T'] / rtn_df['mean_T'].iloc[0]
    rtn_df['rsi_N'] = rtn_df['mean_N'] / rtn_df['mean_N'].iloc[0]
    
    return rtn_df

# Optimal Window

In [24]:
def compute_optimal_window(rtn_df, min_sats_threshold=None):
    """Compute optimal identification window from radial growth rate."""
    if min_sats_threshold:
        rtn_clean = rtn_df[rtn_df['n_sats'] >= min_sats_threshold].copy()
    else:
        rtn_clean = rtn_df.copy()
    
    rtn_clean['radial_growth_rate'] = np.gradient(
        rtn_clean['mean_R'], rtn_clean['days_since_launch']
    )
    
    peak_idx   = rtn_clean['radial_growth_rate'].idxmax()
    peak_day   = rtn_clean.loc[peak_idx, 'days_since_launch']
    peak_rate  = rtn_clean.loc[peak_idx, 'radial_growth_rate']
    half_peak  = peak_rate * 0.5
    below_half = rtn_clean[rtn_clean['radial_growth_rate'] < half_peak]
    
    inflection_day = below_half.iloc[0]['days_since_launch'] if len(below_half) > 0 else None
    
    print(f"Optimal Identification Window:")
    print(f"   Peak growth rate:  {peak_rate:.2f} km/day at day {peak_day}")
    print(f"   50% decay point:   day {inflection_day}")
    print(f"   Optimal window:    days {rtn_clean['days_since_launch'].iloc[0]} — {inflection_day}")
    
    return peak_day, peak_rate, inflection_day, rtn_clean


# Full Pipeline

In [25]:
def run_mission_analysis(intldes, launch_date, session,
                          history_days=60, cluster_focus='C — Upper Core (400-480km)'):
    """
    Run full identification analysis pipeline for a given mission.
    Returns all results as a dictionary.
    """
    print(f"\n{'='*60}")
    print(f"Mission Analysis: {intldes}")
    print(f"Launch Date:      {launch_date}")
    print(f"{'='*60}\n")
    
    # Step 1 — fetch NORAD IDs
    print("Step 1 — Fetching NORAD IDs...")
    norad_ids = fetch_norad_ids(intldes, session)
    
    # Step 2 — fetch current TLEs
    print("\nStep 2 — Fetching current TLEs...")
    tles = fetch_current_tles(intldes, norad_ids, session)
    
    # Step 3 — propagate
    print("\nStep 3 — Propagating orbits...")
    e, r_teme, v_teme, lats, lons, alts, times = propagate_tles(tles)
    
    # Step 4 — build dataframe
    print("\nStep 4 — Building TLE dataframe...")
    df = build_tle_dataframe(tles, alts, e)
    df = cluster_by_altitude(df)
    
    # Step 5 — BSTAR correlation
    print("\nStep 5 — BSTAR correlation...")
    corr, r2 = compute_bstar_correlation(df)
    
    # Step 6 — fetch historical TLEs
    print("\nStep 6 — Fetching historical TLEs...")
    historical_tles = fetch_historical_tles(
        intldes, norad_ids, launch_date, session, days=history_days
    )
    
    # Step 7 — build historical dataframe
    print("\nStep 7 — Building historical dataframe...")
    hist_df = build_hist_dataframe(historical_tles, launch_date)
    
    # Step 8 — filter to focus cluster
    print(f"\nStep 8 — Filtering to {cluster_focus}...")
    cluster_norads = df[df['cluster'] == cluster_focus]['norad'].tolist()
    hist_cluster   = hist_df[hist_df['norad'].isin(cluster_norads)]
    print(f"   {len(cluster_norads)} satellites in focus cluster")
    
    # Step 9 — RTN separation
    print("\nStep 9 — Computing RTN separation...")
    rtn_df = compute_rtn_separation(hist_cluster, launch_date)
    
    # Step 10 — optimal window
    print("\nStep 10 — Computing optimal identification window...")
    min_thresh = int(len(cluster_norads) * 0.85)
    peak_day, peak_rate, inflection_day, rtn_clean = compute_optimal_window(
        rtn_df, min_sats_threshold=min_thresh
    )
    
    print(f"\n{'='*60}")
    print(f"Analysis Complete: {intldes}")
    print(f"   Payloads:          {len(norad_ids)}")
    print(f"   On-orbit TLEs:     {len(tles)}")
    print(f"   BSTAR correlation: {corr:.4f} (R²={r2:.3f})")
    print(f"   Optimal window:    days {rtn_df['days_since_launch'].iloc[0]} — {inflection_day}")
    print(f"{'='*60}\n")
    
    return {
        'intldes':       intldes,
        'launch_date':   launch_date,
        'norad_ids':     norad_ids,
        'tles':          tles,
        'df':            df,
        'hist_df':       hist_df,
        'rtn_df':        rtn_df,
        'rtn_clean':     rtn_clean,
        'corr':          corr,
        'r2':            r2,
        'peak_day':      peak_day,
        'peak_rate':     peak_rate,
        'inflection_day': inflection_day
    }

In [30]:
run_mission_analysis('2023-001', '2023-01-03', session,
                          history_days=60, cluster_focus='C — Upper Core (400-480km)')


Mission Analysis: 2023-001
Launch Date:      2023-01-03

Step 1 — Fetching NORAD IDs...
Querying satcat for 2023-001...
satcat query failed: 401

Step 2 — Fetching current TLEs...
Fetched 0 TLEs for 2023-001

Step 3 — Propagating orbits...
Propagation complete — 0 satellites, 2000 steps
   Errors: 0 non-zero error codes

Step 4 — Building TLE dataframe...


KeyError: 'alt_max'